![butter-eda](../../assets/butter-eda.png)

<a id='top'></a>

### **Title:** BUTTER-E Exploratory Data Analysis — Fully Connected Network Energy Consumption
##### Table of Contents

<ul>
    <li><a href='#data-description'><b>1.0 Dataset Technical Description & Relevance</b></a></li>
    <ul style="margin-top: 5px; margin-bottom: 10px;">
        <li style="margin-left: 20px;"><a href='#data-source'>1.1 Dataset Source and Ownership</a></li>
        <li style="margin-left: 20px;"><a href='#overview'>1.2 Dataset Overview & Key Characteristics</a></li>
    </ul>
    <li><a href='#loading'><b>2.0 Data Loading & Initial Inspection</b></a></li>
    <li><a href='#quality'><b>3.0 Data Quality & Missing Values</b></a></li>
    <li><a href='#distributions'><b>4.0 Feature Distributions</b></a></li>
    <li><a href='#target'><b>5.0 Target Variable Analysis (Energy)</b></a></li>
    <li><a href='#relationships'><b>6.0 Feature-Energy Relationships</b></a></li>
    <li><a href='#hardware'><b>7.0 CPU vs GPU Analysis</b></a></li>
    <li><a href='#summary'><b>8.0 EDA Summary & Key Findings</b></a></li>
</ul>

<a id='data-description'></a>

### 1.0 Dataset Technical Description & Relevanace

<a id='data-source'></a>

#### 1.1 Dataset Source and Ownership

This project uses the [BUTTER-E – Energy Consumption Data for the BUTTER Empirical Deep Learning Dataset](https://data.openei.org/submissions/5991), a large-scale energy benchmark published by the National Renewable Energy Laboratory (NREL) through the <a href="https://data.openei.org/"> Open Energy Data Initiative <img src="../../assets/oedi.png" width="60" alt="OEDI logo"></a>

**Ownership:** The dataset was created by researchers at NREL as part of the BUTTER (Benchmarking Utility for Training Transformations and Empirical Results) project. It is licensed under [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/) and freely available for research use.

The dataset contains energy consumption and performance data from 63,527 individual experimental runs spanning 30,582 distinct configurations of fully connected neural networks (MLPs), measured using node-level hardware watt-meters on both CPU and GPU hardware. It supports the development of machine learning models for predicting neural network training energy consumption from architectural and configuration features, enabling energy-aware decision-making before training begins.

<a id='overview'></a>

#### 1.2 Dataset Overview & Key Characteristics

**Measurement Type: Hardware-Level (Ground Truth)**

BUTTER-E uses node-level hardware watt-meters attached to each compute node on NREL's Eagle HPC system, recording total system power at 1-minute intervals. This captures everything — CPU, GPU, RAM, fans, cooling, and power supply losses. This is ground-truth energy data, unlike software-based tools (e.g., CodeCarbon, Carbontracker) which only read chip-level sensors and are known to underestimate total energy by up to 40% (Fischer, 2025).

**Scale**

* 63,527 individual training runs
* 30,582 distinct hyperparameter configurations
* 12 training datasets × 20 model sizes × 8 network shapes × 14 depths (BUTTER-E's own official OEDI abstract states 13 — a discrepancy in the source documentation; the companion paper's explicit dataset list and this data both show 12)
* Both CPU (Intel Xeon) and GPU (NVIDIA V100) hardware
* Multiple repetitions per configuration (typically 30) to capture variance

**Architecture Type: Fully Connected Networks (MLPs) Only**

All networks in this dataset are dense, fully connected (Multi-Layer Perceptron) architectures. No CNNs, Transformers, or RNNs are included. Convolutional architectures are covered separately by the EC-NAS dataset, which we explore in notebook `01b_ec_nas_eda.ipynb`.

**Files Used in This Project**

| File | Size | Purpose |
|------|------|---------|
| `runs_with_standardized_energy.csv` | 5.21 MB | Main dataset — run metadata joined with energy measurements, quality-filtered and standardized for idle power variation |
| `node_sinfo.csv` | 90 KB | Hardware specs per compute node (CPU cores, memory, GPU type). Joined via `node` column to add hardware features |
| `NVIDIA_GPU_Processors_curated.csv` | 24 KB | GPU metadata (GFLOPS, TDP in watts). Used to extract performance and power features for GPU nodes |
| `pmlb.csv` | 19 KB | Training dataset metadata (n_observations, n_features, n_classes). Joined via `dataset` column to add data complexity features |

**Target Variable**

The primary prediction target is `std_energy` — energy consumption in joules (watt-seconds), computed by integrating the 1-minute power readings over the run's duration, and standardized to correct for varying idle power across compute nodes. Two alternative targets (`energy` for the raw uncorrected measurement, `non_overhead_energy` for overhead-corrected) are available for sensitivity analysis.

**Built-in Quality Control**

The dataset includes pre-computed filters that flag problematic runs:
* `filter_1`: runtime exceeds 75,000 seconds
* `filter_2`: computed energy is zero
* `filter_3`: energy deviates more than 4 standard deviations from similar configurations
* `filter`: logical OR of all three (True = exclude from modeling)

In [1]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# LOADING DATASETS

# Setting the path to where data is stored
DATA_PATH = "../../data/raw/butter_e/"

df = pd.read_csv(DATA_PATH + "runs_with_standardized_energy.csv")
node_info = pd.read_csv(DATA_PATH + "node_sinfo.csv")
gpu_specs = pd.read_csv(DATA_PATH + "NVIDIA_GPU_Processors_curated.csv")
cpu_specs = pd.read_csv(DATA_PATH + "amd_ryzen_processors.csv")


In [3]:
df.head()

,run_id,run_id.1,experiment_id,slurm_job_id,start_time,update_time,node,size,depth,shape,...,start_time_int,start_offset,power,num_reps,std_power,std_energy,energy_overhead,non_overhead_energy,runtime_overhead,non_overhead_runtime
0,0006e91f-3992-42ea-8c60-eaeb0ece0e0e,0006e91f-3992-42ea-8c60-eaeb0ece0e0e,1895249,10430422,2022-10-20 11:20:05.098228+00:00,2022-10-20 11:27:22.441768+00:00,r1i4n33,128,6,wide_first_4x,...,1666264805098228000,2369801375033000,313.681056,4,313.681056,1.371864e+05,14028.665573,1.231577e+05,52.883914,384.459626
1,000a2c53-1754-4082-8c02-c7b3f64de755,000a2c53-1754-4082-8c02-c7b3f64de755,1895553,10370916,2022-10-02 05:22:32.040676+00:00,2022-10-02 05:36:30.882713+00:00,r4i2n20,8388608,5,wide_first_2x,...,1664688152040676000,793148317481000,364.748366,2,364.748366,3.059663e+05,14028.665573,2.919376e+05,52.883914,785.958123
2,001368ba-dcb4-49cd-9ec9-374aa080b2d1,001368ba-dcb4-49cd-9ec9-374aa080b2d1,2039217,10430549,2022-10-20 12:31:17.528985+00:00,2022-10-20 12:35:06.356416+00:00,r6i0n29,4096,14,exponential,...,1666269077528985000,2374073805790000,302.530706,1,302.530706,6.922732e+04,14028.665573,5.519866e+04,52.883914,175.943517
3,00182a76-aa48-483c-ac8f-2743ae150d1e,00182a76-aa48-483c-ac8f-2743ae150d1e,2110207,10430444,2022-10-20 17:45:51.171630+00:00,2022-10-20 23:44:37.495564+00:00,r4i0n30,4194304,20,wide_first_2x,...,1666287951171630000,2392947448435000,331.713663,1,331.713663,7.140576e+06,14028.665573,7.126547e+06,52.883914,21473.440020
4,001b0905-1e1c-4ca3-b0bf-165398cca281,001b0905-1e1c-4ca3-b0bf-165398cca281,1896602,10430377,2022-10-21 01:34:17.322537+00:00,2022-10-21 02:56:22.383860+00:00,r5i2n34,8388608,16,wide_first_8x,...,1666316057322537000,2421053599342000,369.862644,2,369.862644,1.821596e+06,14028.665573,1.807568e+06,52.883914,4872.177409


In [4]:
df.shape

(37055, 33)

In [5]:
df.dtypes

run_id                   object
run_id.1                 object
experiment_id             int64
slurm_job_id              int64
start_time               object
update_time              object
node                     object
size                      int64
depth                     int64
shape                    object
dataset                  object
learning_rate           float64
batch_size                int64
optimizer                object
is_gpu                    int64
batch                    object
energy                  float64
run_time                float64
filter_1                   bool
filter_2                   bool
filter_3                   bool
filter_3_stdevs         float64
filter                     bool
start_time_int            int64
start_offset              int64
power                   float64
num_reps                  int64
std_power               float64
std_energy              float64
energy_overhead         float64
non_overhead_energy     float64
runtime_

In [6]:
df. describe()

,experiment_id,slurm_job_id,size,depth,learning_rate,batch_size,is_gpu,energy,run_time,filter_3_stdevs,start_time_int,start_offset,power,num_reps,std_power,std_energy,energy_overhead,non_overhead_energy,runtime_overhead,non_overhead_runtime
count,3.705500e+04,3.705500e+04,3.705500e+04,37055.000000,3.705500e+04,37055.0,37055.00000,3.705500e+04,37055.000000,37055.000000,3.705500e+04,3.705500e+04,37055.000000,37055.000000,37055.000000,3.705500e+04,37055.000000,3.705500e+04,37055.000000,37055.000000
mean,1.999030e+06,1.038726e+07,1.900497e+06,9.072055,1.000000e-04,256.0,0.39158,1.057974e+06,2758.775821,0.656730,1.665772e+18,1.876563e+15,369.298091,1.594360,365.367717,1.047339e+06,114872.901051,9.324661e+05,289.198485,2469.577336
std,9.526194e+04,3.792951e+04,4.198777e+06,5.301646,1.355271e-20,0.0,0.48811,1.566992e+06,4113.433659,0.539602,9.026080e+14,9.026080e+14,58.875398,1.625017,55.296856,1.556011e+06,125703.784121,1.550834e+06,294.569499,4123.620667
min,1.876157e+06,1.029644e+07,3.200000e+01,2.000000,1.000000e-04,256.0,0.00000,2.998464e+04,114.581339,0.000038,1.663895e+18,0.000000e+00,235.291237,1.000000,230.874162,2.883883e+04,14028.665573,1.481016e+04,52.883914,61.697425
25%,1.897120e+06,1.037088e+07,2.048000e+03,5.000000,1.000000e-04,256.0,0.00000,2.168888e+05,667.340873,0.248270,1.664690e+18,7.948282e+14,322.154378,1.000000,321.341474,2.158241e+05,14028.665573,1.268049e+05,52.883914,329.680548
50%,1.979497e+06,1.037326e+07,6.553600e+04,8.000000,1.000000e-04,256.0,0.00000,4.787344e+05,1242.421294,0.530984,1.666229e+18,2.333988e+15,349.262170,1.000000,348.093224,4.710090e+05,14028.665573,3.363559e+05,52.883914,856.931523
75%,2.094361e+06,1.043041e+07,1.048576e+06,12.000000,1.000000e-04,256.0,1.00000,1.178589e+06,3320.141133,0.933711,1.666299e+18,2.404350e+15,421.948486,1.000000,418.639947,1.173883e+06,271560.240048,1.111264e+06,656.373676,3042.217444
max,2.281610e+06,1.043087e+07,1.677722e+07,20.000000,1.000000e-04,256.0,1.00000,2.215711e+07,67631.756559,3.996610,1.667590e+18,3.694832e+15,579.466998,8.000000,550.466998,2.215711e+07,271560.240048,2.214308e+07,656.373676,67578.872645


In [7]:
key_cols = ['size', 'depth', 'shape', 'dataset', 'learning_rate', 
            'batch_size', 'optimizer', 'is_gpu', 'energy', 
            'std_energy', 'run_time', 'filter']

print("Key columns present:", [c for c in key_cols if c in df.columns])
print("Key columns missing:", [c for c in key_cols if c not in df.columns])

Key columns present: ['size', 'depth', 'shape', 'dataset', 'learning_rate', 'batch_size', 'optimizer', 'is_gpu', 'energy', 'std_energy', 'run_time', 'filter']
Key columns missing: []


<a id='quality'></a>

### 3.0 Data Quality & Missing Values

In [8]:
# METADATA VS ENERGY FILE — resolving the 63,527 (paper) vs 37,055 (loaded) discrepancy

metadata = pd.read_csv(DATA_PATH + "butter_e_metadata.csv")

print("metadata.csv rows:", metadata.shape[0])
print("runs_with_standardized_energy.csv rows:", df.shape[0])
print("energy run_ids also present in metadata:", df['run_id'].isin(metadata['run_id']).sum())

metadata.csv rows: 64988
runs_with_standardized_energy.csv rows: 37055
energy run_ids also present in metadata: 37055


`butter_e_metadata.csv` (64,988 rows) is a superset of run records spanning multiple collection batches; `runs_with_standardized_energy.csv` (37,055 rows) is the fully-matched subset that has energy measurements attached, and is the dataset used going forward. Every row in the energy file has a matching metadata row — nothing is missing or misaligned.

In [9]:
# NULL CHECKS — columns that feed feature engineering directly

feature_cols = ['size', 'depth', 'shape', 'learning_rate', 'batch_size', 'is_gpu']

df[feature_cols].isnull().sum()

size             0
depth            0
shape            0
learning_rate    0
batch_size       0
is_gpu           0
dtype: int64

In [10]:
# SHAPE & DATASET CATEGORIES — what values exist, for family-specific feature planning

print("shape categories:", df['shape'].nunique())
print(df['shape'].value_counts())
print()
print("dataset categories:", df['dataset'].nunique())
print(df['dataset'].value_counts())

shape categories: 8
shape
rectangle             8004
trapezoid             5205
exponential           4880
rectangle_residual    4330
wide_first_2x         3750
wide_first_16x        3645
wide_first_8x         3629
wide_first_4x         3612
Name: count, dtype: int64

dataset categories: 12
dataset
529_pollen             3502
banana                 3484
537_houses             3473
wine_quality_white     3313
294_satellite_image    3229
201_pol                3172
connect_4              3074
nursery                2877
sleep                  2858
splice                 2835
adult                  2762
mnist                  2476
Name: count, dtype: int64


In [11]:
# IS 'size' ALREADY A PARAMETER COUNT, OR DOES IT NEED DERIVING?
# BUTTER's 'size' values are powers of 2 up to ~16.7M -- check that pattern,
# and confirm there's no separate params/FLOPs column already in the data

print("size range:", df['size'].min(), "-", df['size'].max())
print("unique size values:", sorted(df['size'].unique()))
print()
print("columns containing 'param' or 'flop':",
      [c for c in df.columns if 'param' in c.lower() or 'flop' in c.lower()])

size range: 32 - 16777216
unique size values: [np.int64(32), np.int64(64), np.int64(128), np.int64(256), np.int64(512), np.int64(1024), np.int64(2048), np.int64(4096), np.int64(8192), np.int64(16384), np.int64(32768), np.int64(65536), np.int64(131072), np.int64(262144), np.int64(524288), np.int64(1048576), np.int64(2097152), np.int64(4194304), np.int64(8388608), np.int64(16777216)]

columns containing 'param' or 'flop': []


In [12]:
# ENERGY TARGET — confirm units and how energy/power/runtime relate

print(df[['power', 'run_time', 'energy', 'std_power', 'std_energy',
          'non_overhead_energy', 'non_overhead_runtime']].describe())

# sanity check: energy (Wh) should roughly equal power (W) * run_time (s) / 3600
check = (df['power'] * df['run_time'] / 3600 - df['energy']).abs()
print("\nmax abs deviation between power*time and energy:", check.max())

              power      run_time        energy     std_power    std_energy  \
count  37055.000000  37055.000000  3.705500e+04  37055.000000  3.705500e+04   
mean     369.298091   2758.775821  1.057974e+06    365.367717  1.047339e+06   
std       58.875398   4113.433659  1.566992e+06     55.296856  1.556011e+06   
min      235.291237    114.581339  2.998464e+04    230.874162  2.883883e+04   
25%      322.154378    667.340873  2.168888e+05    321.341474  2.158241e+05   
50%      349.262170   1242.421294  4.787344e+05    348.093224  4.710090e+05   
75%      421.948486   3320.141133  1.178589e+06    418.639947  1.173883e+06   
max      579.466998  67631.756559  2.215711e+07    550.466998  2.215711e+07   

       non_overhead_energy  non_overhead_runtime  
count         3.705500e+04          37055.000000  
mean          9.324661e+05           2469.577336  
std           1.550834e+06           4123.620667  
min           1.481016e+04             61.697425  
25%           1.268049e+05       

In [13]:
# CPU vs GPU SPLIT — relevant for keeping hardware comparable to EC-NAS later

df['is_gpu'].value_counts()

is_gpu
0    22545
1    14510
Name: count, dtype: int64

In [17]:
# EPOCHS SANITY CHECK — no 'epochs' column exists in this file (confirmed against
# the BUTTER-E field dictionary: https://github.com/NatLabRockies/BUTTER-E-Empirical-
# analysis-of-energy-trends-in-neural-networks-supplementary-code/blob/main/Readme%20for%20Data.md).
# BUTTER-E only measured energy for the "primary sweep," documented as a fixed
# 3000-epoch budget (Tripp et al.) -- separate from BUTTER's other 300/30k-epoch sweeps.
# If that budget is truly fixed here, repeated runs of the *identical* config should
# show tight run_time clustering (repetition/hardware noise only). Wide spread within
# identical configs would instead suggest a varying epoch count.

group_cols = ['size', 'depth', 'shape', 'dataset', 'learning_rate', 'batch_size', 'optimizer', 'is_gpu']
g = df.groupby(group_cols)['run_time']
multi = g.agg(['count', 'mean', 'std'])
multi = multi[multi['count'] >= 3].copy()
multi['cv'] = multi['std'] / multi['mean']

print(f"configs with >=3 repetitions: {len(multi)} of {g.ngroups}")
print(multi['cv'].describe())
print(f"\nconfigs with CV > 0.5 (would suggest non-constant epoch count): {(multi['cv'] > 0.5).sum()} of {len(multi)}")

configs with >=3 repetitions: 440 of 34585
count    440.000000
mean       0.076735
std        0.123573
min        0.003336
25%        0.012869
50%        0.045448
75%        0.103781
max        1.383039
Name: cv, dtype: float64

configs with CV > 0.5 (would suggest non-constant epoch count): 5 of 440


**Result:** of the 440 configs with ≥3 repetitions, `run_time` coefficient of variation is tiny (median ≈4.5%, 75th percentile ≈10%); only 5 of 440 (1.1%) exceed CV 0.5, plausibly hardware contention outliers rather than a varying epoch count. This is consistent with a fixed epoch budget — repeated runs of the same config finish in essentially the same time, which wouldn't hold if epoch count varied per run.

**Decision:** `epochs = 3000` is locked in as a documented, fixed constant for the BUTTER-E feature table (footnote: primary-sweep-only energy measurement uses a fixed 3000-epoch budget per Tripp et al.; not independently confirmed at the per-run level since epoch-level data would require the much larger `summary_by_epoch.tar`, but the run_time consistency check above supports it). Not pulling `summary_by_epoch.tar` for now — revisit only if something downstream suggests epoch count actually varies.

<a id='summary'></a>

### 8.0 EDA Summary & Key Findings

EDA scoped to what Phase 1 (shared feature table) needs — no key columns have nulls, `shape`/`dataset` categories are enumerated (12, not the 13 in BUTTER-E's own official abstract — a documented source-side discrepancy), `size` is confirmed as parameter count (not something to derive), the energy target (`std_energy`, in joules not watt-hours — corrected from the initial writeup) and its relationship to power/runtime is confirmed, and the CPU/GPU split is known. Two feature-engineering inputs were resolved as design decisions rather than data lookups: FLOPs will be approximated as `2 × size` for now (no per-layer widths available; upgrade later only if it hurts accuracy), and `epochs` will be locked in as a documented constant `3000` (primary-sweep fixed budget, supported by tight within-config `run_time` clustering above — not independently confirmed at the per-run level). Full distribution plots, correlation heatmaps, and outlier visualization are deferred to Section 5 thesis figures — not blocking. Next: `02a_butter_e_features.ipynb`.

![butter-eda](../../assets/thanks-eda.png)